In [1]:
import pandas as pd
from training.static_data import StaticData
from datetime import date

myDate = date(2024, 6, 15)
folder_path = "../../../../data"
model_path = "../../../../models"

koda_static_path = f"{folder_path}/koda-static/data-tmp"

# Load data similarly to api
try:
    static_data = StaticData.load_from_pkl(koda_static_path)
except FileNotFoundError:
    static_data = StaticData.load_static_data(koda_static_path)

    static_data.save_to_pkl(koda_static_path)

route_short_name = "4"
SL_AGENCY_ID = static_data.agencies[static_data.agencies["agency_name"] == "AB Storstockholms Lokaltrafik"].index[0]
route_subset = static_data.routes[
    (static_data.routes['route_short_name'] == route_short_name) & \
    (static_data.routes["agency_id"] == SL_AGENCY_ID)]

if not route_subset.empty:
    route_id = route_subset.index[0]
    print(f"Route ID: {route_id}")
    
    trips_for_route = static_data.trips[static_data.trips["route_id"] == route_id]
    print(f"Trips count: {len(trips_for_route)}")
    
    trip_ids = trips_for_route.index
    
    # Filter stop times
    relevant_stop_times = static_data.stop_times[
        static_data.stop_times.index.get_level_values(0).isin(trip_ids)
    ].sort_index()
    
    # Check sequences
    sequences = relevant_stop_times.groupby("trip_id")["stop_id"].apply(tuple)
    print("Top sequences:")
    print(sequences.value_counts().head())
    
    most_common = sequences.value_counts().index[0]
    print(f"Most common length: {len(most_common)}")

    # Resolve stops
    stops_indexed = static_data.stops.set_index("stop_id")
    for sid in most_common[:5]:
        print(stops_indexed.loc[sid])
else:
    print("Route not found")


Route ID: 9011001000400000
Trips count: 1809
Top sequences:
stop_id
(9022001011725004, 9022001010261002, 9022001010401001, 9022001010136002, 9022001010661002, 9022001010658001, 9022001010655002, 9022001010651002, 9022001010649002, 9022001010645002, 9022001010369001, 9022001010367002, 9022001010363004, 9022001010183002, 9022001010151003, 9022001010188002, 9022001010191002, 9022001010193001, 9022001010194004, 9022001010199002, 9022001010198002, 9022001010045002, 9022001010203001, 9022001010627002, 9022001010098003)                                                          455
(9022001010098003, 9022001010627001, 9022001010203002, 9022001010045001, 9022001010040001, 9022001010198001, 9022001010199001, 9022001010194006, 9022001010192001, 9022001010191001, 9022001010188001, 9022001010151002, 9022001010183003, 9022001010363005, 9022001010367001, 9022001010369008, 9022001010421001, 9022001010645001, 9022001010647001, 9022001010649001, 9022001010651001, 9022001010655001, 9022001010657001, 90220

In [2]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from datetime import date
from training.common import FeedID, Operator

operator = Operator.SL
myDate = date(2024, 6, 15)
hour = 10
feedId = FeedID.TripUpdates

In [5]:
from training.koda import download_koda_rt_file, download_koda_static_file

KODA_API_KEY = os.getenv("KODA_API_KEY")
if not KODA_API_KEY:
    raise ValueError("KODA_API_KEY not found in environment variables")

download_koda_rt_file(operator, feedId, myDate, api_key=KODA_API_KEY, data_dir=f"{folder_path}/koda-rt")
download_koda_rt_file(operator, FeedID.VehiclePositions, myDate, api_key=KODA_API_KEY, data_dir=f"{folder_path}/koda-rt")

download_koda_static_file(operator, myDate, api_key=KODA_API_KEY, data_dir=f"{folder_path}/koda-static")

File ../../../../data/koda-rt/sl_2024-06-15_TripUpdates.7z already exists. Skipping download.
File ../../../../data/koda-rt/sl_2024-06-15_VehiclePositions.7z already exists. Skipping download.
File ../../../../data/koda-static/sl_2024-06-15.zip already exists. Skipping download.


200

In [6]:
import pickle
from training.static_data import StaticData

static_data: StaticData
pickle_file_path = f"{folder_path}/koda-static/static-data-{myDate.year:04d}-{myDate.month:02d}-{myDate.day:02d}.pkl"
if os.path.exists(pickle_file_path):
    print("Static data pickle file exists.")

    static_data = pickle.load(open(pickle_file_path, "rb"))
else:
    print("Loading static data from GTFS files.")
    static_data = StaticData.load_static_data(f"{folder_path}/koda-static/data-tmp")
    pickle.dump(static_data, open(pickle_file_path, "wb"))
static_data

Static data pickle file exists.


In [7]:
print(static_data.stop_times.dtypes)
static_data.stop_times.head()

stop_id                                   string[python]
stop_headsign                             string[python]
pickup_type                                     category
drop_off_type                                   category
shape_dist_traveled                              float64
timepoint                                          int64
arrival_time_seconds_since_midnight      timedelta64[ns]
departure_time_seconds_since_midnight    timedelta64[ns]
dtype: object


stop_id   stop_headsign pickup_type  \
trip_id           stop_sequence                                                 
14010000631461521 1              9022001010369006  Stora Essingen           3   
                  2              9022001010421001  Stora Essingen           3   
                  3              9022001010645003  Stora Essingen           3   
                  4              9022001010462002  Stora Essingen           3   
                  5              9022001010449001  Stora Essingen           3   

                                drop_off_type  shape_dist_traveled  timepoint  \
trip_id           stop_sequence                                                 
14010000631461521 1                         1                 0.00          1   
                  2                         3               430.80          0   
                  3                         3               828.25          0   
                  4                         3              1032.70          0   
                  5                         3              1327.70          0   

                                arrival_time_seconds_since_midnight  \
trip_id           stop_sequence                                       
14010000631461521 1                                 0 days 07:30:00   
                  2                                 0 days 07:31:02   
                  3                                 0 days 07:32:14   
                  4                                 0 days 07:32:51   
                  5                                 0 days 07:33:44   

                                departure_time_seconds_since_midnight  
trip_id           stop_sequence                                        
14010000631461521 1                                   0 days 07:30:00  
                  2                                   0 days 07:31:02  
                  3                                   0 days 07:32:14  
                  4                                   0 days 07:32:51  
                  5                                   0 days 07:33:44

In [8]:
from training.gtfs import  load_pb_file

koda_rt_example_path = f"{folder_path}/koda-rt/data-tmp/sl/{feedId.value}/{myDate.year:04d}/{myDate.month:02d}/{myDate.day:02d}/{hour:02d}"

files_in_data_folder = os.listdir(koda_rt_example_path)
print("Files in data folder:", files_in_data_folder)

first_file_path = os.path.join(koda_rt_example_path, files_in_data_folder[0])

gtfs_feed_message = load_pb_file(first_file_path)

Files in data folder: ['sl-tripupdates-2024-06-15T10-00-03Z.pb', 'sl-tripupdates-2024-06-15T10-00-17Z.pb', 'sl-tripupdates-2024-06-15T10-00-31Z.pb', 'sl-tripupdates-2024-06-15T10-00-45Z.pb', 'sl-tripupdates-2024-06-15T10-01-13Z.pb', 'sl-tripupdates-2024-06-15T10-01-27Z.pb', 'sl-tripupdates-2024-06-15T10-01-41Z.pb', 'sl-tripupdates-2024-06-15T10-01-55Z.pb', 'sl-tripupdates-2024-06-15T10-02-09Z.pb', 'sl-tripupdates-2024-06-15T10-02-23Z.pb', 'sl-tripupdates-2024-06-15T10-02-37Z.pb', 'sl-tripupdates-2024-06-15T10-02-51Z.pb', 'sl-tripupdates-2024-06-15T10-03-05Z.pb', 'sl-tripupdates-2024-06-15T10-03-19Z.pb', 'sl-tripupdates-2024-06-15T10-03-33Z.pb', 'sl-tripupdates-2024-06-15T10-03-47Z.pb', 'sl-tripupdates-2024-06-15T10-04-01Z.pb', 'sl-tripupdates-2024-06-15T10-04-15Z.pb', 'sl-tripupdates-2024-06-15T10-04-29Z.pb', 'sl-tripupdates-2024-06-15T10-04-43Z.pb', 'sl-tripupdates-2024-06-15T10-05-11Z.pb', 'sl-tripupdates-2024-06-15T10-05-25Z.pb', 'sl-tripupdates-2024-06-15T10-05-39Z.pb', 'sl-tripupd

In [9]:
from training.data_processing import feed_message_to_trip_update_dataframe

df = feed_message_to_trip_update_dataframe(gtfs_feed_message)

print(df.dtypes)

print("Number of rows in DataFrame:", len(df))

df.head()

id                                Int64
trip_id                  string[python]
start_date               datetime64[ns]
schedule_relationship             int64
vehicle_id                        Int64
stop_time_updates                object
timestamp                         int64
dtype: object
Number of rows in DataFrame: 689


,id,trip_id,start_date,schedule_relationship,vehicle_id,stop_time_updates,timestamp
0,14010516247726276,14010000663741929,2024-06-15,0,9031008000500536,"[{'stop_sequence': 25, 'stop_id': '90220010001...",1718438398
1,14010516247729020,14010000663747674,2024-06-15,0,9031008000500539,"[{'stop_sequence': 11, 'stop_id': '90220010001...",1718438398
2,14010516089523031,14010000656705623,2024-06-15,0,9031001004002220,"[{'stop_sequence': 24, 'stop_id': '90220010050...",1718438398
3,14010516113220145,14010000656788822,2024-06-15,0,9031001004302521,"[{'stop_sequence': 21, 'stop_id': '90220010061...",1718438398
4,14010516089526040,14010000656705875,2024-06-15,0,9031001004002221,"[{'stop_sequence': 21, 'stop_id': '90220010051...",1718438398


In [10]:
from training.data_processing import join_static_data_on_rt_trip_updates


df = join_static_data_on_rt_trip_updates(static_data, df)

In [11]:
df[(df["route_short_name"] == "40") & (df["route_desc"] == "Pendeltåg")].head()

,id,trip_id,start_date,schedule_relationship,vehicle_id,stop_time_updates,timestamp,route_id,service_id,trip_headsign,direction_id,shape_id,agency_id,route_short_name,route_long_name,route_type,route_desc
2,14010516089523031,14010000656705623,2024-06-15,0,9031001004002220,"[{'stop_sequence': 24, 'stop_id': '90220010050...",1718438398,9011001004000000,6,<NA>,0.0,4014010000492969507,14010000000001001,40,<NA>,100,Pendeltåg
4,14010516089526040,14010000656705875,2024-06-15,0,9031001004002221,"[{'stop_sequence': 21, 'stop_id': '90220010051...",1718438398,9011001004000000,332,<NA>,1.0,4014010000492969547,14010000000001001,40,<NA>,100,Pendeltåg
19,14010516089527825,14010000656705974,2024-06-15,0,9031001004002222,"[{'stop_sequence': 17, 'stop_id': '90220010050...",1718438398,9011001004000000,433,<NA>,0.0,4014010000492969316,14010000000001001,40,<NA>,100,Pendeltåg
25,14010516089529610,14010000656706199,2024-06-15,0,9031001004002223,"[{'stop_sequence': 12, 'stop_id': '90220010053...",1718438398,9011001004000000,332,<NA>,1.0,4014010000492969547,14010000000001001,40,<NA>,100,Pendeltåg
113,14010516089531701,14010000656706391,2024-06-15,0,9031001004002224,"[{'stop_sequence': 8, 'stop_id': '902200100516...",1718438398,9011001004000000,6,<NA>,0.0,4014010000492969507,14010000000001001,40,<NA>,100,Pendeltåg


In [12]:
from training.data_processing import explode_to_stops_with_join_static

# print(df.dtypes)

# print(static_data.stop_times.dtypes)

df_exploded_with_stop_times = explode_to_stops_with_join_static(static_data, df)

df_exploded_with_stop_times.head()

datetime64[ns]


id start_date  \
trip_id           stop_sequence                                 
14010000663741929 25             14010516247726276 2024-06-15   
                  26             14010516247726276 2024-06-15   
                  27             14010516247726276 2024-06-15   
                  28             14010516247726276 2024-06-15   
                  29             14010516247726276 2024-06-15   

                                 schedule_relationship        vehicle_id  \
trip_id           stop_sequence                                            
14010000663741929 25                                 0  9031008000500536   
                  26                                 0  9031008000500536   
                  27                                 0  9031008000500536   
                  28                                 0  9031008000500536   
                  29                                 0  9031008000500536   

                                  timestamp          route_id  service_id  \
trip_id           stop_sequence                                             
14010000663741929 25             1718438398  9011008001300000           6   
                  26             1718438398  9011008001300000           6   
                  27             1718438398  9011008001300000           6   
                  28             1718438398  9011008001300000           6   
                  29             1718438398  9011008001300000           6   

                                trip_headsign  direction_id  \
trip_id           stop_sequence                               
14010000663741929 25                     <NA>           0.0   
                  26                     <NA>           0.0   
                  27                     <NA>           0.0   
                  28                     <NA>           0.0   
                  29                     <NA>           0.0   

                                            shape_id          agency_id  \
trip_id           stop_sequence                                           
14010000663741929 25             6014010000657796244  14010000000002071   
                  26             6014010000657796244  14010000000002071   
                  27             6014010000657796244  14010000000002071   
                  28             6014010000657796244  14010000000002071   
                  29             6014010000657796244  14010000000002071   

                                route_short_name route_long_name  route_type  \
trip_id           stop_sequence                                                
14010000663741929 25                          13            <NA>        1000   
                  26                          13            <NA>        1000   
                  27                          13            <NA>        1000   
                  28                          13            <NA>        1000   
                  29                          13            <NA>        1000   

                                      route_desc           stop_id  \
trip_id           stop_sequence                                      
14010000663741929 25             Waxholmsbolaget  9022001000143001   
                  26             Waxholmsbolaget  9022001000142001   
                  27             Waxholmsbolaget  9022001000141001   
                  28             Waxholmsbolaget  9022001000139001   
                  29             Waxholmsbolaget  9022001000138001   

                                       arrival_time      departure_time  \
trip_id           stop_sequence                                           
14010000663741929 25            2024-06-15 07:51:16 2024-06-15 07:52:13   
                  26            2024-06-15 07:53:16 2024-06-15 07:53:16   
                  27            2024-06-15 07:55:06 2024-06-15 07:55:49   
                  28            2024-06-15 07:59:29 2024-06-15 07:59:29   
                  29            2024-06-15 07:59:

In [13]:
# df_exploded_with_stop_times["arrival_time_late"] = (df_exploded_with_stop_times["arrival_time"] - df_exploded_with_stop_times["arrival_time_planned"])
# df_exploded_with_stop_times["departure_time_late"] = (df_exploded_with_stop_times["departure_time"] - df_exploded_with_stop_times["departure_time_planned"])

print(df_exploded_with_stop_times[["arrival_time", "arrival_time_planned", "arrival_time_late",
                             "departure_time", "departure_time_planned", "departure_time_late", "agency_id"]].dtypes)

df_exploded_with_stop_times[["arrival_time", "arrival_time_planned", "arrival_time_late",
                             "departure_time", "departure_time_planned", "departure_time_late", "agency_id"]].head()

arrival_time               datetime64[ns]
arrival_time_planned       datetime64[ns]
arrival_time_late         timedelta64[ns]
departure_time             datetime64[ns]
departure_time_planned     datetime64[ns]
departure_time_late       timedelta64[ns]
agency_id                  string[python]
dtype: object


arrival_time arrival_time_planned  \
trip_id           stop_sequence                                            
14010000663741929 25            2024-06-15 07:51:16  2024-06-15 07:46:00   
                  26            2024-06-15 07:53:16  2024-06-15 07:47:00   
                  27            2024-06-15 07:55:06  2024-06-15 07:50:00   
                  28            2024-06-15 07:59:29  2024-06-15 07:53:00   
                  29            2024-06-15 07:59:59  2024-06-15 07:54:00   

                                arrival_time_late      departure_time  \
trip_id           stop_sequence                                         
14010000663741929 25              0 days 00:05:16 2024-06-15 07:52:13   
                  26              0 days 00:06:16 2024-06-15 07:53:16   
                  27              0 days 00:05:06 2024-06-15 07:55:49   
                  28              0 days 00:06:29 2024-06-15 07:59:29   
                  29              0 days 00:05:59 2024-06-15 08:00:01   

                                departure_time_planned departure_time_late  \
trip_id           stop_sequence                                              
14010000663741929 25               2024-06-15 07:46:00     0 days 00:06:13   
                  26               2024-06-15 07:47:00     0 days 00:06:16   
                  27               2024-06-15 07:50:00     0 days 00:05:49   
                  28               2024-06-15 07:53:00     0 days 00:06:29   
                  29               2024-06-15 07:54:00     0 days 00:06:01   

                                         agency_id  
trip_id           stop_sequence                     
14010000663741929 25             14010000000002071  
                  26             14010000000002071  
                  27             14010000000002071  
                  28             14010000000002071  
                  29             14010000000002071

In [14]:
SL_AGENCY_ID = static_data.agencies[static_data.agencies["agency_name"] == "AB Storstockholms Lokaltrafik"].index[0]
# WAXHOLMSBOLAGET_ID = static_data.agencies[static_data.agencies["agency_name"] == "Waxholmsbolaget Ångfartygs AB"].index[0]

# route_dt = df_exploded_with_stop_times[
#   (df_exploded_with_stop_times["route_short_name"] == "13") & \
#   (df_exploded_with_stop_times["agency_id"] == WAXHOLMSBOLAGET_ID)]
# route_dt.head(20)

In [15]:
df_exploded_with_stop_times[(df_exploded_with_stop_times["route_desc"] == "blåbuss") & \
  (df_exploded_with_stop_times["route_short_name"].isin(["1", "2", "3", "4"])) & \
  (df_exploded_with_stop_times["agency_id"] == SL_AGENCY_ID)]["route_short_name"].value_counts()

route_short_name
4    170
2    107
1     69
3     53
Name: count, dtype: Int64

In [16]:
df_exploded_with_stop_times[(df_exploded_with_stop_times["route_desc"] == "blåbuss") & \
  (df_exploded_with_stop_times["route_short_name"].isin(["4"])) & \
  (df_exploded_with_stop_times["agency_id"] == SL_AGENCY_ID)].head(20)

id start_date  \
trip_id           stop_sequence                                 
14010000637145800 21             14010515937867392 2024-06-15   
                  22             14010515937867392 2024-06-15   
                  23             14010515937867392 2024-06-15   
                  24             14010515937867392 2024-06-15   
                  25             14010515937867392 2024-06-15   
14010000641290683 15             14010515950004186 2024-06-15   
                  16             14010515950004186 2024-06-15   
                  17             14010515950004186 2024-06-15   
                  18             14010515950004186 2024-06-15   
                  19             14010515950004186 2024-06-15   
                  20             14010515950004186 2024-06-15   
                  21             14010515950004186 2024-06-15   
                  22             14010515950004186 2024-06-15   
                  23             14010515950004186 2024-06-15   
                  24             14010515950004186 2024-06-15   
                  25             14010515950004186 2024-06-15   
14010000641290864 16             14010515949994700 2024-06-15   
                  17             14010515949994700 2024-06-15   
                  18             14010515949994700 2024-06-15   
                  19             14010515949994700 2024-06-15   

                                 schedule_relationship        vehicle_id  \
trip_id           stop_sequence                                            
14010000637145800 21                                 0  9031001001004010   
                  22                                 0  9031001001004010   
                  23                                 0  9031001001004010   
                  24                                 0  9031001001004010   
                  25                                 0  9031001001004010   
14010000641290683 15                                 0  9031001001001555   
                  16                                 0  9031001001001555   
                  17                                 0  9031001001001555   
                  18                                 0  9031001001001555   
                  19                                 0  9031001001001555   
                  20                                 0  9031001001001555   
                  21                                 0  9031001001001555   
                  22                                 0  9031001001001555   
                  23                                 0  9031001001001555   
                  24                                 0  9031001001001555   
                  25                                 0  9031001001001555   
14010000641290864 16                                 0  9031001001001553   
                  17                                 0  9031001001001553   
                  18                                 0  9031001001001553   
                  19                                 0  9031001001001553   

                                  timestamp          route_id  service_id  \
trip_id           stop_sequence                                             
14010000637145800 21             1718438398  9011001000400000           6   
                  22             1718438398  9011001000400000           6   
                  23             1718438398  9011001000400000           6   
                  24             1718438398  9011001000400000           6   
                  25             1718438398  9011001000400000           6   
14010000641290683 15             1718438398  9011001000400000           6   
                  16             1718438398  9011001000400000           6   
                  17             1718438398  9011001000400000           6   
                  18             1718438398  9011001000400000           6   
                  19             1718438398  9011001000400000           6   
                  20

In [17]:
df_exploded_with_stop_times = df_exploded_with_stop_times.sort_values(['trip_id', 'stop_sequence'])

df_exploded_with_stop_times['arrival_time_prev'] = df_exploded_with_stop_times.groupby('trip_id')['arrival_time'].shift(1)
df_exploded_with_stop_times['departure_time_prev'] = df_exploded_with_stop_times.groupby('trip_id')['departure_time'].shift(1)
df_exploded_with_stop_times['arrival_time_planned_prev'] = df_exploded_with_stop_times.groupby('trip_id')['arrival_time_planned'].shift(1)
df_exploded_with_stop_times['departure_time_planned_prev'] = df_exploded_with_stop_times.groupby('trip_id')['departure_time_planned'].shift(1)
df_exploded_with_stop_times['arrival_time_late_prev'] = df_exploded_with_stop_times.groupby('trip_id')['arrival_time_late'].shift(1)
df_exploded_with_stop_times['departure_time_late_prev'] = df_exploded_with_stop_times.groupby('trip_id')['departure_time_late'].shift(1)

df_exploded_with_stop_times['arrival_time_next'] = df_exploded_with_stop_times.groupby('trip_id')['arrival_time'].shift(-1)
df_exploded_with_stop_times['departure_time_next'] = df_exploded_with_stop_times.groupby('trip_id')['departure_time'].shift(-1)
df_exploded_with_stop_times['arrival_time_planned_next'] = df_exploded_with_stop_times.groupby('trip_id')['arrival_time_planned'].shift(-1)
df_exploded_with_stop_times['departure_time_planned_next'] = df_exploded_with_stop_times.groupby('trip_id')['departure_time_planned'].shift(-1)
df_exploded_with_stop_times['arrival_time_late_next'] = df_exploded_with_stop_times.groupby('trip_id')['arrival_time_late'].shift(-1)
df_exploded_with_stop_times['departure_time_late_next'] = df_exploded_with_stop_times.groupby('trip_id')['departure_time_late'].shift(-1)
df_exploded_with_stop_times.head(40)

id start_date  \
trip_id           stop_sequence                                 
14010000496968823 1              14010515507510490 2024-06-15   
                  2              14010515507510490 2024-06-15   
                  3              14010515507510490 2024-06-15   
                  4              14010515507510490 2024-06-15   
                  5              14010515507510490 2024-06-15   
                  6              14010515507510490 2024-06-15   
                  7              14010515507510490 2024-06-15   
                  8              14010515507510490 2024-06-15   
                  9              14010515507510490 2024-06-15   
                  10             14010515507510490 2024-06-15   
                  11             14010515507510490 2024-06-15   
                  12             14010515507510490 2024-06-15   
                  13             14010515507510490 2024-06-15   
                  14             14010515507510490 2024-06-15   
14010000496968868 1              14010515507510606 2024-06-15   
                  2              14010515507510606 2024-06-15   
                  3              14010515507510606 2024-06-15   
                  4              14010515507510606 2024-06-15   
                  5              14010515507510606 2024-06-15   
                  6              14010515507510606 2024-06-15   
                  7              14010515507510606 2024-06-15   
                  8              14010515507510606 2024-06-15   
                  9              14010515507510606 2024-06-15   
                  10             14010515507510606 2024-06-15   
                  11             14010515507510606 2024-06-15   
                  12             14010515507510606 2024-06-15   
                  13             14010515507510606 2024-06-15   
                  14             14010515507510606 2024-06-15   
14010000502525650 6              14010515507510258 2024-06-15   
                  7              14010515507510258 2024-06-15   
                  8              14010515507510258 2024-06-15   
                  9              14010515507510258 2024-06-15   
                  10             14010515507510258 2024-06-15   
                  11             14010515507510258 2024-06-15   
                  12             14010515507510258 2024-06-15   
                  13             14010515507510258 2024-06-15   
                  14             14010515507510258 2024-06-15   
14010000502525695 1              14010515507510374 2024-06-15   
                  2              14010515507510374 2024-06-15   
                  3              14010515507510374 2024-06-15   

                                 schedule_relationship        vehicle_id  \
trip_id           stop_sequence                                            
14010000496968823 1                                  0  9031001002510015   
                  2                                  0  9031001002510015   
                  3                                  0  9031001002510015   
                  4                                  0  9031001002510015   
                  5                                  0  9031001002510015   
                  6                                  0  9031001002510015   
                  7                                  0  9031001002510015   
                  8                                  0  9031001002510015   
                  9                                  0  9031001002510015   
                  10                                 0  9031001002510015   
                  11                                 0  9031001002510015   
                  12                                 0  9031001002510015   
                  13                                 0  9031001002510015   
                  14                                 0  9031001002510015   
14010000496968868 1                                  0  9031001002510003   
                  2      

In [18]:
from training.gtfs import download_gtfs_static_file

GTFS_REGIONAL_STATIC_API_KEY = os.getenv("GTFS_REGIONAL_STATIC_API_KEY")
if not GTFS_REGIONAL_STATIC_API_KEY:
    raise ValueError("GTFS_REGIONAL_STATIC_API_KEY not found in environment variables")

download_gtfs_static_file(Operator.SL, api_key=GTFS_REGIONAL_STATIC_API_KEY, data_dir=f"{folder_path}/gtfs-static")

File ../../../../data/gtfs-static/sl_gtfs_static.zip already exists. Skipping download.


In [19]:
from training.gtfs import download_gtfs_rt_file

GTFS_REGIONAL_RT_API_KEY = os.getenv("GTFS_REGIONAL_RT_API_KEY")
if not GTFS_REGIONAL_RT_API_KEY:
    raise ValueError("GTFS_REGIONAL_RT_API_KEY not found in environment variables")

download_gtfs_rt_file(Operator.SL, FeedID.TripUpdates, api_key=GTFS_REGIONAL_RT_API_KEY, data_dir=f"{folder_path}/gtfs-rt")
download_gtfs_rt_file(Operator.SL, FeedID.VehiclePositions, api_key=GTFS_REGIONAL_RT_API_KEY, data_dir=f"{folder_path}/gtfs-rt")

File ../../../../data/gtfs-rt/sl_TripUpdates.pb exists. Sending If-Modified-Since: Sun, 11 Jan 2026 23:05:52 GMT
Downloaded and saved GTFS-RT feed to ../../../../data/gtfs-rt/sl_TripUpdates.pb.
File ../../../../data/gtfs-rt/sl_VehiclePositions.pb exists. Sending If-Modified-Since: Sun, 11 Jan 2026 23:05:52 GMT
Downloaded and saved GTFS-RT feed to ../../../../data/gtfs-rt/sl_VehiclePositions.pb.


In [20]:
todayDate = date.today()

gtfs_static_path = f"{folder_path}/gtfs-static/data-tmp"

# Load data similarly to api
try:
    gtfs_static_data = StaticData.load_from_pkl(gtfs_static_path)
except FileNotFoundError:
    gtfs_static_data = StaticData.load_static_data(gtfs_static_path)

    gtfs_static_data.save_to_pkl(gtfs_static_path)

In [21]:
first_file_path = os.path.join(folder_path, "gtfs-rt/sl_VehiclePositions.pb")

gtfs_feed_message = load_pb_file(first_file_path)

In [22]:
from training.data_processing import feed_message_to_vehicle_position_dataframe, join_static_data_on_rt_vehicle_positions

df = feed_message_to_vehicle_position_dataframe(gtfs_feed_message)
df = join_static_data_on_rt_vehicle_positions(gtfs_static_data, df)

print(df.dtypes)

print("Number of rows in DataFrame:", len(df))

df.head()

id                                       Int64
trip_id                         string[python]
timestamp                                int64
vehicle_latitude                       float64
vehicle_longitude                      float64
vehicle_bearing                        float64
vehicle_odometer                       float64
vehicle_speed                          float64
vehicle_congestion_level                 int64
vehicle_occupancy_percentage             int64
vehicle_occupancy_status                 int64
stop_id                                 object
current_status                           int64
current_stop_sequence                    int64
route_id                                object
service_id                               Int64
trip_headsign                   string[python]
direction_id                           float64
shape_id                                 Int64
agency_id                       string[python]
route_short_name                string[python]
route_long_na

,id,trip_id,timestamp,vehicle_latitude,vehicle_longitude,vehicle_bearing,vehicle_odometer,vehicle_speed,vehicle_congestion_level,vehicle_occupancy_percentage,vehicle_occupancy_status,stop_id,current_status,current_stop_sequence,route_id,service_id,trip_headsign,direction_id,shape_id,agency_id,route_short_name,route_long_name,route_type,route_desc
0,888801768172765420,14010100669283559,1768172766,59.346802,18.070055,0.0,0.0,-0.3,0,0,0,,2,0,9011001002900000,46,<NA>,1.0,2014010000351290634,14010000000001001,29,Näsbyparkslinjen,900,Roslagsbanan
1,701768172352704,14010000702413332,1768172766,59.236858,18.098932,149.0,0.0,-0.3,0,0,0,,2,0,9011001001800000,35,<NA>,0.0,3014010000563704255,14010000000001001,18,Gröna linjen,401,tunnelbanans gröna linje
2,53631768172765962,14010000704108294,1768172766,59.316528,18.237286,54.0,0.0,9.7,0,0,0,,2,0,9011001042200000,221,<NA>,1.0,1014010000703448208,14010000000001001,422,<NA>,700,<NA>
3,72091768172765165,14010000647498260,1768172766,59.172337,17.437702,170.0,0.0,9.7,0,0,0,,2,0,9011001078000000-780X,46,<NA>,0.0,1014010000618661979,14010000000001001,780X,<NA>,700,<NA>
4,41331768172765425,14010000698648821,1768172766,59.414265,17.931126,353.0,0.0,8.3,0,0,0,,2,0,9011001017900000,213,<NA>,1.0,1014010000369798744,14010000000001001,179,<NA>,700,blåbuss


In [23]:
SL_AGENCY_ID = gtfs_static_data.agencies[gtfs_static_data.agencies["agency_name"] == "AB Storstockholms Lokaltrafik"].index[0]

STAM_BUSES = ["1", "2", "3", "4"]

df[df["route_short_name"].isin(STAM_BUSES) & (df["agency_id"] == SL_AGENCY_ID)]["route_short_name"].value_counts()

route_short_name
4    7
2    4
3    4
1    4
Name: count, dtype: Int64

In [24]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor()
xgb_model.load_model(f"{model_path}/9011001001300000_xgb_model.json")


In [25]:
import datetime
import dotenv
from training.data_pipeline import create_X_from_df, lag_times
from training.gtfs import load_gtfs_rt_immediately
from training.data_processing import  feed_message_to_vehicle_position_dataframe, join_static_data_on_rt_vehicle_positions
import training.__generated__.gtfs_realtime_pb2 as gtfs_rt
from training.common import FeedID, Operator
dotenv.load_dotenv()

GTFS_RT_API_KEY = os.environ.get("GTFS_REGIONAL_RT_API_KEY", "")
if GTFS_RT_API_KEY == "":
    raise Exception("GTFS_REGIONAL_RT_API_KEY environment variable not set")

gtfs_static_data = StaticData.load_static_data(f"{folder_path}/gtfs-static/data-tmp")

gtfs_vehicle_positions = load_gtfs_rt_immediately(
    operator=Operator.SL, 
    feedId=FeedID.VehiclePositions, 
    api_key= GTFS_RT_API_KEY,
)
# gtfs_trip_updates = load_gtfs_rt_immediately(
#     operator=Operator.SL, 
#     feedId=FeedID.TripUpdates, 
#     api_key= GTFS_RT_API_KEY,
# )

def debug_print_feed_entity(feed_message: gtfs_rt.FeedEntity) -> None:
    print(feed_message)

debug_print_feed_entity(gtfs_vehicle_positions.entity[0])

df_vehicles = feed_message_to_vehicle_position_dataframe(gtfs_vehicle_positions)
# df_vehicles = join_static_data_on_rt_vehicle_positions(gtfs_static_data, df_vehicles)

print(df_vehicles.dtypes)

df_vehicles.head()

# df_vehicles = lag_times(df_vehicles)

# x = create_X_from_df(gtfs_vehicle_positions)

# y = xgb_model.predict(x)

# gtfs_vehicle_positions["next_stop_estimated_arrival_time"] = y


id: "888801768172778421"
vehicle {
  trip {
    trip_id: "14010100669283559"
    schedule_relationship: SCHEDULED
  }
  position {
    latitude: 59.3468
    longitude: 18.070055
    speed: -0.3
  }
  timestamp: 1768172778
  vehicle {
    id: "9031001009988880"
  }
}

id                                       Int64
trip_id                         string[python]
timestamp                                int64
vehicle_latitude                       float64
vehicle_longitude                      float64
vehicle_bearing                        float64
vehicle_odometer                       float64
vehicle_speed                          float64
vehicle_congestion_level                 int64
vehicle_occupancy_percentage             int64
vehicle_occupancy_status                 int64
stop_id                                 object
current_status                           int64
current_stop_sequence                    int64
dtype: object


,id,trip_id,timestamp,vehicle_latitude,vehicle_longitude,vehicle_bearing,vehicle_odometer,vehicle_speed,vehicle_congestion_level,vehicle_occupancy_percentage,vehicle_occupancy_status,stop_id,current_status,current_stop_sequence
0,888801768172778421,14010100669283559,1768172778,59.346802,18.070055,0.0,0.0,-0.3,0,0,0,,2,0
1,701768172352704,14010000702413332,1768172778,59.236858,18.098932,149.0,0.0,-0.3,0,0,0,,2,0
2,53631768172778021,14010000704108294,1768172778,59.317253,18.238832,47.0,0.0,10.3,0,0,0,,2,0
3,72091768172778418,14010000647498260,1768172778,59.171112,17.438190,169.0,0.0,10.8,0,0,0,,2,0
4,41331768172778385,14010000698648821,1768172778,59.415287,17.932348,38.0,0.0,11.4,0,0,0,,2,0


In [26]:
import datetime
import os

def list_gtfs_files(base_path: str, start_dt: datetime.datetime, end_dt: datetime.datetime):
    found_files = []

    if start_dt.tzinfo is None:
        start_dt = start_dt.replace(tzinfo=datetime.timezone.utc)
    if end_dt.tzinfo is None:
        end_dt = end_dt.replace(tzinfo=datetime.timezone.utc)

    if not os.path.exists(base_path):
        return []

    with os.scandir(base_path) as years:
        for year_entry in years:
            if not year_entry.is_dir() or not year_entry.name.isdigit():
                continue
            
            year = int(year_entry.name)
            if not (start_dt.year <= year <= end_dt.year):
                continue

            with os.scandir(year_entry.path) as months:
                for month_entry in months:
                    if not month_entry.is_dir() or not month_entry.name.isdigit():
                        continue
                    
                    month = int(month_entry.name)
                    if year == start_dt.year and month < start_dt.month:
                        continue
                    if year == end_dt.year and month > end_dt.month:
                        continue

                    # 3. DAY Level
                    with os.scandir(month_entry.path) as days:
                        for day_entry in days:
                            if not day_entry.is_dir() or not day_entry.name.isdigit():
                                continue
                            
                            day = int(day_entry.name)
                            
                            current_date = datetime.date(year, month, day)
                            
                            if current_date < start_dt.date() or current_date > end_dt.date():
                                continue

                            with os.scandir(day_entry.path) as hours:
                                for hour_entry in hours:
                                    if not hour_entry.is_dir() or not hour_entry.name.isdigit():
                                        continue
                                    
                                    hour = int(hour_entry.name)
                                    
                                    # Lower bound check
                                    if current_date == start_dt.date() and hour < start_dt.hour:
                                        continue
                                    # Upper bound check
                                    if current_date == end_dt.date() and hour > end_dt.hour:
                                        continue
                                    with os.scandir(hour_entry.path) as files:
                                        for file_entry in files:
                                            # Parse timestamp from filename
                                            ts_str = file_entry.name.split('tripupdates-')[-1].replace('.pb', '')
                                            
                                            # Standard parsing
                                            file_dt = datetime.datetime.strptime(ts_str, "%Y-%m-%dT%H-%M-%SZ")
                                            file_dt = file_dt.replace(tzinfo=datetime.timezone.utc)
                                            
                                            if start_dt <= file_dt <= end_dt:
                                                found_files.append(file_entry.path)

    return found_files

now = datetime.datetime.now(datetime.timezone.utc)
ten_minute_files = list_gtfs_files(
    base_path=f"{folder_path}/gtfs-rt/data-tmp/sl/TripUpdates",
    start_dt=now - datetime.timedelta(minutes=10),
    end_dt=now
)

ten_minute_files

['../../../../data/gtfs-rt/data-tmp/sl/TripUpdates/2026/01/11/22/sl-tripupdates-2026-01-11T22-56-22Z.pb',
 '../../../../data/gtfs-rt/data-tmp/sl/TripUpdates/2026/01/11/22/sl-tripupdates-2026-01-11T22-56-37Z.pb',
 '../../../../data/gtfs-rt/data-tmp/sl/TripUpdates/2026/01/11/22/sl-tripupdates-2026-01-11T22-56-52Z.pb',
 '../../../../data/gtfs-rt/data-tmp/sl/TripUpdates/2026/01/11/22/sl-tripupdates-2026-01-11T22-57-07Z.pb',
 '../../../../data/gtfs-rt/data-tmp/sl/TripUpdates/2026/01/11/22/sl-tripupdates-2026-01-11T22-57-22Z.pb',
 '../../../../data/gtfs-rt/data-tmp/sl/TripUpdates/2026/01/11/22/sl-tripupdates-2026-01-11T22-57-37Z.pb',
 '../../../../data/gtfs-rt/data-tmp/sl/TripUpdates/2026/01/11/22/sl-tripupdates-2026-01-11T22-57-53Z.pb',
 '../../../../data/gtfs-rt/data-tmp/sl/TripUpdates/2026/01/11/22/sl-tripupdates-2026-01-11T22-58-08Z.pb',
 '../../../../data/gtfs-rt/data-tmp/sl/TripUpdates/2026/01/11/22/sl-tripupdates-2026-01-11T22-58-23Z.pb',
 '../../../../data/gtfs-rt/data-tmp/sl/TripUpd

In [28]:
from training.gtfs import load_pb_file
from training.data_processing import feed_message_to_trip_update_dataframe, join_static_data_on_rt_trip_updates, explode_to_stops_with_join_static

gtfs_static_data = StaticData.load_static_data(f"{folder_path}/gtfs-static/data-tmp")

def load_gtfs_frame_from_files(
    file_paths: list[str],
    gtfs_static_data: StaticData
) -> pd.DataFrame:
    trips = []
    for file_path in file_paths:
        gtfs_feed_message = load_pb_file(file_path)
        trip_update_df = feed_message_to_trip_update_dataframe(gtfs_feed_message)
        trip_update_df = join_static_data_on_rt_trip_updates(gtfs_static_data, trip_update_df)
        trips.append(trip_update_df)

    trips = pd.concat(trips, ignore_index=True)
    return trips

trips = load_gtfs_frame_from_files(ten_minute_files, gtfs_static_data)
trips = trips[trips["route_short_name"].isin(["1", "2", "3", "4"])]

print(gtfs_static_data.stop_times.dtypes)

trips = explode_to_stops_with_join_static(gtfs_static_data, trips)
trips = lag_times(trips)

# Only keep the last line stop_sequence per trip_id
# multiindex: trip_id	stop_sequence	
trips = trips.sort_values(['trip_id', 'stop_sequence'])
trips = trips.groupby(level='trip_id').last()

trips

stop_id                                   string[python]
stop_headsign                             string[python]
pickup_type                                     category
drop_off_type                                   category
shape_dist_traveled                              float64
timepoint                                          int64
pickup_booking_rule_id                           float64
drop_off_booking_rule_id                         float64
arrival_time_seconds_since_midnight      timedelta64[ns]
departure_time_seconds_since_midnight    timedelta64[ns]
dtype: object
datetime64[ns]


,id,start_date,schedule_relationship,vehicle_id,timestamp,route_id,service_id,trip_headsign,direction_id,shape_id,agency_id,route_short_name,route_long_name,route_type,route_desc,stop_id,arrival_time,departure_time,stop_time_schedule_relationship,stop_id_planned,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,pickup_booking_rule_id,drop_off_booking_rule_id,arrival_time_seconds_since_midnight,departure_time_seconds_since_midnight,arrival_time_planned,departure_time_planned,arrival_time_late,departure_time_late,arrival_time_prev,departure_time_prev,arrival_time_planned_prev,departure_time_planned_prev,arrival_time_late_prev,departure_time_late_prev
trip_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
14010000637042868,14010517577069619,2026-01-11,0,9031001001001553,1768172774,9011001000300000,14,<NA>,1.0,1014010000571984641,14010000000001001,3,<NA>,700,blåbuss,9022001010406001,2026-01-11 23:20:14,2026-01-11 23:20:14,0,9022001010406001,Södersjukhuset,1,3,10297.95,1,NaN,NaN,1 days 00:24:00,1 days 00:24:00,2026-01-11 22:24:00,2026-01-11 22:24:00,0 days 00:56:14,0 days 00:56:14,2026-01-11 23:18:58,2026-01-11 23:18:58,2026-01-11 22:19:55,2026-01-11 22:19:55,0 days 00:59:03,0 days 00:59:03
14010000664274440,14010517577071761,2026-01-11,0,9031001001007205,1768172774,9011001000300000,14,<NA>,1.0,1014010000571984641,14010000000001001,3,<NA>,700,blåbuss,9022001010406001,2026-01-11 23:05:54,2026-01-11 23:05:54,0,9022001010406001,Södersjukhuset,1,3,10297.95,1,NaN,NaN,1 days 00:06:00,1 days 00:06:00,2026-01-11 22:06:00,2026-01-11 22:06:00,0 days 00:59:54,0 days 00:59:54,2026-01-11 23:04:49,2026-01-11 23:04:49,2026-01-11 22:01:14,2026-01-11 22:01:14,0 days 01:03:35,0 days 01:03:35
14010000664274475,14010517576911175,2026-01-11,0,9031001001001549,1768172774,9011001000300000,14,<NA>,0.0,1014010000573107346,14010000000001001,3,<NA>,700,blåbuss,9022001050800003,2026-01-11 23:14:49,2026-01-11 23:15:18,0,9022001050800003,Karolinska sjukhuset,1,3,9861.36,1,NaN,NaN,1 days 00:10:00,1 days 00:10:00,2026-01-11 22:10:00,2026-01-11 22:10:00,0 days 01:04:49,0 days 01:05:18,2026-01-11 23:13:00,2026-01-11 23:13:08,2026-01-11 22:05:02,2026-01-11 22:05:02,0 days 01:07:58,0 days 01:08:06
14010000664274517,14010517576911959,2026-01-11,0,9031001001007218,1768172774,9011001000300000,14,<NA>,0.0,1014010000573107346,14010000000001001,3,<NA>,700,blåbuss,9022001050800003,2026-01-11 23:26:04,2026-01-11 23:26:04,0,9022001050800003,Karolinska sjukhuset,1,3,9861.36,1,NaN,NaN,1 days 00:30:00,1 days 00:30:00,2026-01-11 22:30:00,2026-01-11 22:30:00,0 days 00:56:04,0 days 00:56:04,2026-01-11 23:25:26,2026-01-11 23:25:26,2026-01-11 22:25:02,2026-01-11 22:25:02,0 days 01:00:24,0 days 01:00:24
14010000664378596,14010517572835328,2026-01-11,0,9031001001004002,1768172774,9011001000400000,7,<NA>,0.0,1014010000631454799,14010000000001001,4,<NA>,700,blåbuss,9022001010098003,2026-01-11 23:19:33,2026-01-11 23:20:21,0,9022001010098003,Radiohuset,1,3,12259.83,1,NaN,NaN,1 days 00:19:00,1 days 00:19:00,2026-01-11 22:19:00,2026-01-11 22:19:00,0 days 01:00:33,0 days 01:01:21,2026-01-11 23:18:42,2026-01-11 23:18:47,2026-01-11 22:17:00,2026-01-11 22:17:00,0 days 01:01:42,0 days 01:01:47
14010000664378857,14010517573390331,2026-01-11,0,9031001001004001,1768172774,9011001000100000,7,<NA>,0.0,1014010000560262054,14010000000001001,1,<NA>,700,blåbuss,9022001010028001,2026-01-11 23:34:09,2026-01-11 23:35:18,0,9022001010028001,Frihamnen,1,3,10834.28,1,NaN,NaN,1 days 00:37:00,1 days 00:37:00,2026-01-11 22:37:00,2026-01-11 22:37:00,0 days 00:57:09,0 days 00:58:18,2026-01-11 23:33:35,2026-01-11 23:33:45,2026-01-11 22:35:18,2026-01-11 22:35:18,0 days 00:58:17,0 days 00:58:27
14010000666494551,14010517578537857,2026-01-11,0,9031001001004017,1768172419,9011001000100000,10,<NA>,0.0,1014010000560262054,14010000000001001,1,<NA>,700,blåbuss,9022001010028001,2026-01-11 23:00:03,2026-01-11 23:00:03,0,9022001010028001,Frihamnen,1,3,10834.28,1,NaN,NaN,1 days 00:00:00,1 days 00:00:00,2026-01-11 22:0

In [29]:
trips["route_short_name"].value_counts()

route_short_name
4    7
3    4
1    4
2    4
Name: count, dtype: Int64

In [30]:
import dotenv
from training.gtfs import load_gtfs_rt_immediately
from training.data_processing import  feed_message_to_vehicle_position_dataframe
from training.common import FeedID, Operator
dotenv.load_dotenv()

GTFS_RT_API_KEY = os.environ.get("GTFS_REGIONAL_RT_API_KEY", "")
if GTFS_RT_API_KEY == "":
    raise Exception("GTFS_REGIONAL_RT_API_KEY environment variable not set")

gtfs_vehicle_positions = load_gtfs_rt_immediately(
    operator=Operator.SL, 
    feedId=FeedID.VehiclePositions, 
    api_key= GTFS_RT_API_KEY,
)

df_vehicles = feed_message_to_vehicle_position_dataframe(gtfs_vehicle_positions)

df_vehicles

,id,trip_id,timestamp,vehicle_latitude,vehicle_longitude,vehicle_bearing,vehicle_odometer,vehicle_speed,vehicle_congestion_level,vehicle_occupancy_percentage,vehicle_occupancy_status,stop_id,current_status,current_stop_sequence
0,888801768172837419,14010100669283559,1768172837,59.346802,18.070055,0.0,0.0,-0.3,0,0,0,,2,0
1,701768172352704,14010000702413332,1768172837,59.236858,18.098932,149.0,0.0,-0.3,0,0,0,,2,0
2,53631768172836925,14010000704108294,1768172837,59.319958,18.247545,67.0,0.0,9.2,0,0,0,,2,0
3,72091768172836811,14010000647498260,1768172837,59.166523,17.442495,129.0,0.0,8.1,0,0,0,,2,0
4,41331768172837393,14010000698648821,1768172837,59.419468,17.941511,36.0,0.0,11.4,0,0,0,,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436,1411768172823888,14010000702413627,1768172837,59.301739,18.079830,345.0,0.0,-0.3,0,0,0,,2,0
437,1321768172769123,14010000702413752,1768172837,59.323498,18.066910,152.0,0.0,-0.3,0,0,0,,2,0
438,4291768172836705,14010000711194074,1768172837,59.627407,17.722219,72.0,0.0,6.1,0,0,0,,2,0
439,72211768172836922,14010000710270644,1768172837,59.306393,18.077242,162.0,0.0,9.7,0,0,0,,2,0


In [34]:
vehicles_with_trips = df_vehicles.join(trips, on="trip_id", rsuffix="_trip", how="inner")

vehicles_with_trips

,id,trip_id,timestamp,vehicle_latitude,vehicle_longitude,vehicle_bearing,vehicle_odometer,vehicle_speed,vehicle_congestion_level,vehicle_occupancy_percentage,vehicle_occupancy_status,stop_id,current_status,current_stop_sequence,id_trip,start_date,schedule_relationship,vehicle_id,timestamp_trip,route_id,service_id,trip_headsign,direction_id,shape_id,agency_id,route_short_name,route_long_name,route_type,route_desc,stop_id_trip,arrival_time,departure_time,stop_time_schedule_relationship,stop_id_planned,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,pickup_booking_rule_id,drop_off_booking_rule_id,arrival_time_seconds_since_midnight,departure_time_seconds_since_midnight,arrival_time_planned,departure_time_planned,arrival_time_late,departure_time_late,arrival_time_prev,departure_time_prev,arrival_time_planned_prev,departure_time_planned_prev,arrival_time_late_prev,departure_time_late_prev
41,72161768172291013,14010000710270606,1768172837,59.299103,18.080441,66.0,0.0,2.5,0,0,0,,2,0,14010517615688133,2026-01-11,0,9031001001007216,1768172292,9011001000400000,14,<NA>,1.0,1014010000710251654,14010000000001001,4,<NA>,700,blåbuss,9022001011725007,2026-01-11 22:57:49,2026-01-11 22:57:49,0,9022001011725007,Gullmarsplan,1,3,12245.68,1,NaN,NaN,0 days 23:57:00,0 days 23:57:00,2026-01-11 21:57:00,2026-01-11 21:57:00,0 days 01:00:49,0 days 01:00:49,2026-01-11 22:54:10,2026-01-11 22:54:31,2026-01-11 21:51:00,2026-01-11 21:51:00,0 days 01:03:10,0 days 01:03:31
75,40291768172836808,14010000702332193,1768172837,59.317951,18.054489,257.0,0.0,4.2,0,0,0,,2,0,14010517572835379,2026-01-11,0,9031001001004029,1768172774,9011001000400000,7,<NA>,0.0,1014010000631454799,14010000000001001,4,<NA>,700,blåbuss,9022001010098003,2026-01-11 23:30:50,2026-01-11 23:31:00,0,9022001010098003,Radiohuset,1,3,12259.83,1,NaN,NaN,1 days 00:33:00,1 days 00:33:00,2026-01-11 22:33:00,2026-01-11 22:33:00,0 days 00:57:50,0 days 00:58:00,2026-01-11 23:30:00,2026-01-11 23:30:00,2026-01-11 22:31:00,2026-01-11 22:31:00,0 days 00:59:00,0 days 00:59:00
177,71621768172836248,14010000697800531,1768172837,59.318462,18.079237,108.0,0.0,6.1,0,0,0,,2,0,14010517576236873,2026-01-11,0,9031001001007162,1768172774,9011001000200000,14,<NA>,1.0,1014010000686241947,14010000000001001,2,<NA>,700,blåbuss,9022001010881003,2026-01-11 23:11:41,2026-01-11 23:11:41,0,9022001010881003,Sofia,1,3,7553.76,1,NaN,NaN,1 days 00:12:00,1 days 00:12:00,2026-01-11 22:12:00,2026-01-11 22:12:00,0 days 00:59:41,0 days 00:59:41,2026-01-11 23:10:57,2026-01-11 23:11:02,2026-01-11 22:09:45,2026-01-11 22:09:45,0 days 01:01:12,0 days 01:01:17
199,40301768172443873,14010000702331897,1768172837,59.335545,18.099684,112.0,0.0,0.0,0,0,0,,2,0,14010517572835277,2026-01-11,0,9031001001004030,1768172454,9011001000400000,7,<NA>,0.0,1014010000631454799,14010000000001001,4,<NA>,700,blåbuss,9022001010098003,2026-01-11 23:00:43,2026-01-11 23:02:00,0,9022001010098003,Radiohuset,1,3,12259.83,1,NaN,NaN,1 days 00:04:00,1 days 00:04:00,2026-01-11 22:04:00,2026-01-11 22:04:00,0 days 00:56:43,0 days 00:58:00,2026-01-11 22:59:41,2026-01-11 22:59:56,2026-01-11 22:02:00,2026-01-11 22:02:00,0 days 00:57:41,0 days 00:57:56
209,15531768172837030,14010000637042868,1768172837,59.327431,18.064257,141.0,0.0,8.9,0,0,0,,2,0,14010517577069619,2026-01-11,0,9031001001001553,1768172774,9011001000300000,14,<NA>,1.0,1014010000571984641,14010000000001001,3,<NA>,700,blåbuss,9022001010406001,2026-01-11 23:20:14,2026-01-11 23:20:14,0,9022001010406001,Södersjukhuset,1,3,10297.95,1,NaN,NaN,1 days 00:24:00,1 days 00:24:00,2026-01-11 22:24:00,2026-01-11 22:24:00,0 days 00:56:14,0 days 00:56:14,2026-01-11 23:18:58,2026-01-11 23:18:58,2026-01-11 22:19:55,2026-01-11 22:19:55,0 days 00:59:03,0 days 00:59:03
226,72151768172836808,14010000710270682,1768172837,59.332218,18.030130,280.0,0.0,6.4,0,0,0,,2,0,14010517615680432,2026-01-11,0,9031001001007215,1768172774,9011001000400000,14,<NA>,1.0,1014010000710251654,14010000000001001,4,<NA>,700,blåbuss,9022001011

In [ ]:
vehicles_with_trips[["arrival_time_planned", "arrival_time_planned_prev", "arrival_time_late_prev"]
]

,arrival_time_planned,arrival_time_planned_prev,arrival_time_late_prev
41,2026-01-11 21:57:00,2026-01-11 21:51:00,0 days 01:03:10
75,2026-01-11 22:33:00,2026-01-11 22:31:00,0 days 00:59:00
177,2026-01-11 22:12:00,2026-01-11 22:09:45,0 days 01:01:12
199,2026-01-11 22:04:00,2026-01-11 22:02:00,0 days 00:57:41
209,2026-01-11 22:24:00,2026-01-11 22:19:55,0 days 00:59:03
226,2026-01-11 22:27:00,2026-01-11 22:22:00,0 days 00:59:33
246,2026-01-11 22:33:00,2026-01-11 22:31:28,0 days 00:58:03
251,2026-01-11 22:10:00,2026-01-11 22:05:02,0 days 01:07:58
256,2026-01-11 22:13:00,2026-01-11 22:09:58,0 days 01:01:28
264,2026-01-11 22:06:00,2026-01-11 22:01:14,0 days 01:03:35


In [39]:
# x = create_X_from_df(vehicles_with_trips)

# pred = xgb_model.predict(x)

# vehicles_with_trips["next_stop_estimated_arrival_time"] = pred

# vehicles_with_trips[["trip_id", "stop_id", "arrival_time_planned",
#             "arrival_time_planned_prev",
#             "arrival_time_late_prev", "next_stop_estimated_arrival_time"]]